<a href="https://colab.research.google.com/github/natdebandi/hate_speech_ar/blob/main/4_analysis_hate_ar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Analisis del Dataset completo marzo 2020 - abril 2021 de X

**Natalia Dedandi**

Se aplica el clasificador seleccionado a todo el conjunto de datos recoelctado para analizar las características del odio en Argentina durante la pandemia

In [1]:
%pip install datasets seaborn


   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 8.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/618.0 kB ? eta -:--:--
   --------------------------------------- 618.0/618.0 kB 11.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ------- -------------------------------- 5.5/27.6 MB 30.5 MB/s eta 0:00:01
   ----------------- ---------------------- 11.8/27.6 MB 29.5 MB/s eta 0:00:01
   ----------------------- ---------------- 16.0/27.6 MB 25.8 MB/s eta 0:00:01
   ------------------------------- -------- 21.5/27.6 MB 25.6 MB/s eta 0:00:01
   ---------------------------------------  27.5/27.6 MB 26.8 MB/s eta 0:00:01
   ---------------------------------------- 27.6/27.6 MB 24.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 27.3 MB/s eta 0:00:00
Note: you


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install pandas
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Recupero el conjutno completo de datos: https://huggingface.co/datasets/piuba-bigdata/articles_and_comments



In [3]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd

load_dotenv() 
token = os.getenv("HF_TOKEN")


dataset = load_dataset("piubamas/articles_and_comments", token=token)

c:\Users\natal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\natal\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\natal\.cache\huggingface\hub\datasets--piubamas--articles_and_comments. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

In [5]:
#from datasets import load_dataset
#import pandas as pd

#dataset = load_dataset("piubamas/articles_and_comments")


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tweet_id', 'text', 'title', 'url', 'user', 'body', 'created_at', 'comments'],
        num_rows: 537201
    })
})

In [7]:
ds1 = dataset["train"]

Ejemplo de una fila

In [9]:
ds1[354]

{'tweet_id': '1377046872343392256',
 'text': 'Lawfare, el día después https://t.co/QDZnv0QbX4',
 'title': None,
 'url': None,
 'user': 'pagina12',
 'body': None,
 'created_at': '2021-03-30T23:55:33Z',
 'comments': [{'created_at': '2021-03-30T23:58:03Z',
   'prediction': {'APPEARANCE': 0,
    'CALLS': 0,
    'CLASS': 0,
    'CRIMINAL': 0,
    'DISABLED': 0,
    'LGBTI': 0,
    'POLITICS': 0,
    'RACISM': 0,
    'WOMEN': 0},
   'text': '@pagina12 https://t.co/LfVPBEvLR0',
   'tweet_id': '1377047501866405893',
   'user_id': '89962248'},
  {'created_at': '2021-03-30T23:58:43Z',
   'prediction': {'APPEARANCE': 0,
    'CALLS': 0,
    'CLASS': 0,
    'CRIMINAL': 0,
    'DISABLED': 0,
    'LGBTI': 0,
    'POLITICS': 0,
    'RACISM': 0,
    'WOMEN': 0},
   'text': '@pagina12 Y dale con temas que a nadie importa, salvo a la mechera. Mientras tanto los precios al galope. Inútiles',
   'tweet_id': '1377047670540398592',
   'user_id': '1349538511670673408'},
  {'created_at': '2021-03-30T23:58:50Z'

En este conjunto de datos tengo las etiquetas de la clasificación realizada con el modelo BETO:

https://huggingface.co/piuba-bigdata/beto-contextualized-hate-speech

Puede usarse para comparar pero en principio nos interesa recuperar los comentarios completos para aplicarle el clasificador seleccionado

In [10]:
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from collections import Counter

In [11]:
tweets_arg = []

for noticia in tqdm(dataset['train']):

    date_noticia = noticia["created_at"]
    # Convertimos la fecha de la noticia a un objeto de python
    datenoticia = datetime.strptime(date_noticia, "%Y-%m-%dT%H:%M:%S%fZ")
    i=0
    for comment in noticia["comments"]:
        date_comment = comment["created_at"]
        # Convertimos la fecha del comentario a un objeto de python
        datecomment = datetime.strptime(date_comment, "%Y-%m-%dT%H:%M:%S%fZ")
        # anexa comentarios de diarios argentinos
        tweets_arg.append({"id":i,
            "tweet_id_noticia": noticia["tweet_id"],
            "title": noticia["title"],
            "resumen": noticia["text"],
            "medio": noticia["user"],
            "date_tweet": datecomment,
            "text": comment["text"],
            "user_id": comment["user_id"],
            "CALLS":comment["prediction"]["CALLS"],
            "WOMEN":comment["prediction"]["WOMEN"],
            "LGBTI":comment["prediction"]["LGBTI"],
            "RACISM":comment["prediction"]["RACISM"],
            "CLASS":comment["prediction"]["CLASS"],
            "POLITICS":comment["prediction"]["POLITICS"],
            "DISABLED":comment["prediction"]["DISABLED"],
            "APPEARANCE":comment["prediction"]["APPEARANCE"],
            "CRIMINAL":comment["prediction"]["CRIMINAL"]
            })
        i=i+1

100%|██████████| 537201/537201 [08:04<00:00, 1109.35it/s]


In [13]:
tweets_arg[1156]

{'id': 150,
 'tweet_id_noticia': '1376946868161220610',
 'title': 'Polémica: un proyecto K propone multas millonarias y endurecer penas de prisión para el que propague el Covid',
 'resumen': 'Polémica: un proyecto K propone multas millonarias y endurecer penas de prisión para el que propague el Covid https://t.co/snNm3UBvmR',
 'medio': 'clarincom',
 'date_tweet': datetime.datetime(2021, 3, 30, 18, 9, 0, 800000),
 'text': '@clarincom Cómo en el funeral que organizaron del Diego. O las marchas y contramarchas del aborto',
 'user_id': '882377587',
 'CALLS': 0,
 'WOMEN': 0,
 'LGBTI': 0,
 'RACISM': 0,
 'CLASS': 0,
 'POLITICS': 0,
 'DISABLED': 0,
 'APPEARANCE': 0,
 'CRIMINAL': 0}

In [14]:
# prompt: create a dataframe from tweets_arg

df_tweets_arg = pd.DataFrame(tweets_arg)


In [15]:
df_tweets_arg['HATEFUL']= df_tweets_arg[['CALLS','WOMEN','LGBTI','RACISM','CLASS','POLITICS','DISABLED','APPEARANCE','CRIMINAL']].max(axis=1)

In [16]:
df_tweets_arg.head()

,id,tweet_id_noticia,title,resumen,medio,date_tweet,text,user_id,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL,HATEFUL
0,0,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,1532596098,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,582286194,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,951552761988034560,0,0,0,0,0,0,0,0,0,0


coloco el mismo orden de las etiquetas que usa el clasificador

In [17]:
# prompt: change the order of columns of df_tweets_arg and put HATEFUL before CALLS


df_tweets_arg = df_tweets_arg[['id', 'tweet_id_noticia','title','resumen','medio', 'date_tweet', 'text', 'user_id', 'HATEFUL', 'CALLS', 'WOMEN',
       'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE',
       'CRIMINAL']]
df_tweets_arg.head()

,id,tweet_id_noticia,title,resumen,medio,date_tweet,text,user_id,HATEFUL,CALLS,WOMEN,LGBTI,RACISM,CLASS,POLITICS,DISABLED,APPEARANCE,CRIMINAL
0,0,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:03:00.900,@clarincom A mi me preocupa el trabajo.. La ev...,1532596098,0,0,0,0,0,0,0,0,0,0
1,1,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:05:04.500,@clarincom Lo que preocupa. https://t.co/Vmf9V...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
2,2,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:06:03.100,@clarincom Lo que les preocupa. https://t.co/P...,1222094021687943168,0,0,0,0,0,0,0,0,0,0
3,3,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:11:02.300,@clarincom Le recomendaríamos al presidente de...,582286194,0,0,0,0,0,0,0,0,0,0
4,4,1376940813968609288,Segunda ola de coronavirus: preocupan las reun...,Segunda ola de coronavirus: preocupan las reun...,clarincom,2021-03-30 17:26:00.600,@clarincom Para salvar al correo de la quiebra...,951552761988034560,0,0,0,0,0,0,0,0,0,0


In [18]:
# prompt: save df_tweets_arg to csv file

df_tweets_arg.to_csv('data/tweets_con_noticia.csv', index=False)


In [17]:
# prompt: len of df_tweets_arg

len(df_tweets_arg)


8569648